# Train the quantile transformer (any GPU notebook: Kaggle / Colab / Studio Lab)
Run all cells top to bottom. The setup cell clones the repo itself if it isn't already inside one, so a fresh Kaggle or Colab session needs no manual steps — on Kaggle, switch Session options to a GPU accelerator and turn **Internet** on first.

**Kaggle GPU choice: use T4 ×2, not the P100.** The P100 is a Pascal card (compute capability sm_60) that current PyTorch wheels no longer ship kernels for — training crashes at the first kernel launch. The T4 (sm_75) is fully supported; the second T4 simply idles.

After a session cutoff (idle disconnect, GPU-quota limit), just Run All again: completed runs skip themselves, the interrupted run auto-resumes from its last checkpoint, and any runs that haven't started yet train fresh. No flags or manual bookkeeping needed between sessions.

Config changes are detected automatically: if you edit a config (e.g. a different `train.lr`) after a run already trained against it, the next Run All notices the saved config no longer matches and retrains that run from scratch on its own — no flag needed. `--fresh` remains available in a cell's command to force a retrain of an UNCHANGED config (e.g. just to redo a run). `--resume` is still accepted for backward compatibility but is now a no-op — auto-resume is the default.

In [ ]:
import os, pathlib, subprocess
root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if root is None:  # fresh Kaggle/Colab session: not inside a clone yet, so make one
    subprocess.run(["git", "clone", "https://github.com/mtsilverstein/Megatron.git"], check=True)
    root = pathlib.Path.cwd() / "Megatron"
os.chdir(root)
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"
import torch; print('cuda:', torch.cuda.is_available())

## Hyperparameter sweep (optional, before committing to a walk-forward config)

Tunes ONLY on the `configs/transformer_v1_through2022.yaml` fold (train <= 2021,
val = 2022) -- held-out test seasons 2023-2025 are never touched here; the
sweep module does not import or run the eval harness at all (see
`src/ffmodel/model/sweep.py`'s module docstring for the full binding
protocol).

**Resumable for free:** if this Studio Lab session's 4h GPU quota runs out
mid-sweep, just re-run the next cell -- completed combos are skipped, the
combo that was mid-training auto-resumes from its last epoch checkpoint,
and combos that haven't started yet train fresh. No flags, no manual
bookkeeping.

**Expected wall time:** the default grid (`configs/sweep_v1.yaml`) is 24
combos; each combo is a full short training run (a handful to dozens of
epochs with early stopping), so budget on the order of a few minutes per
combo on a T4 (more for the larger `d_model`/`seq_len` combos) -- likely
more than one GPU session for the full grid. That's fine: see above.

In [ ]:
!python -m ffmodel.model.sweep --grid configs/sweep_v1.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
import json
from pathlib import Path

import pandas as pd

results = json.loads(Path("models/sweeps/v1/results.json").read_text())
# collect_results() already sorts best (lowest val_pinball) first and
# flags any not-yet-complete combo -- this is just a readable view of it.
pd.json_normalize(results)

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2022.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2023.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1_through2024.yaml --features-parquet data/features_2012_2025.parquet

In [ ]:
!python -m ffmodel.model.train --config configs/transformer_v1.yaml --features-parquet data/features_2012_2025.parquet

## Seed ensemble (optional)

Trains seeds `43` and `44` over **all four** walk-forward configs
(`transformer_v1_through2022.yaml`, `_through2023.yaml`, `_through2024.yaml`,
and the production `transformer_v1.yaml`) — 8 short runs total — so the
resulting ensemble can be honestly walk-forward evaluated by the bake-off
below, not just deployed on faith. Each seed lands in its own sibling
artifact dir (`v1_s43/through*`, `v1_s44/through*`) alongside the unsuffixed
`v1/through*` runs, never overwriting them.

Skip-if-complete (same as every other train cell above) makes re-running
this cell free after a session cutoff or a config edit — only the runs
that changed or haven't finished retrain.

**This cell is entirely optional.** Skip it and the bake-off cell below
just evaluates the single main `v1` run, exactly as before. Run it, and the
bake-off cell automatically discovers whichever seed roots finished
completely and evaluates the ensemble instead — no flags to flip.

Once you're happy with the bake-off numbers, deploying the ensemble on the
live site (instead of the single seed) is a config change, not a code
change: set `.github/workflows/weekly-update.yml`'s `ARTIFACT_ROOT` to the
comma-separated roots (e.g.
`models/transformer/v1_s43,models/transformer/v1_s44`) — see the README's
"Deployed model" section for the full contract.

In [ ]:
import subprocess

configs = [
    "configs/transformer_v1_through2022.yaml",
    "configs/transformer_v1_through2023.yaml",
    "configs/transformer_v1_through2024.yaml",
    "configs/transformer_v1.yaml",
]

for seed in (43, 44):
    for cfg in configs:
        subprocess.run(
            ["python", "-m", "ffmodel.model.train", "--config", cfg,
             "--features-parquet", "data/features_2012_2025.parquet",
             "--seed", str(seed)],
            check=True,
        )

## Promoting a sweep winner

Once `models/sweeps/v1/results.json` has a winner you're happy with (lowest
`val_pinball`, `complete: true`), turn it into the walk-forward configs the
bake-off below actually uses:

1. Copy the winning combo's `params` (from its `results.json` row) into
   **all four** `configs/transformer_v1*.yaml` files
   (`transformer_v1_through2022.yaml`, `transformer_v1_through2023.yaml`,
   `transformer_v1_through2024.yaml`, and the production
   `transformer_v1.yaml`) -- dotted keys like `model.d_model` map directly
   onto those files' `model:`/`train:` blocks. No `--fresh` needed after
   editing: the train cells above detect that each config changed and
   retrain those runs from scratch automatically on the next Run All.
2. Optionally train a full seed ensemble on the winning config using the
   "Seed ensemble (optional)" cell above -- it covers all four configs (not
   just production), so the ensemble itself gets an honest walk-forward
   bake-off rather than just a production-fold artifact.
3. Nothing else to do at eval time: the bake-off cell below auto-discovers
   whatever complete ensemble roots exist and evaluates them together. To
   deploy the ensemble on the live site instead of the single seed, see the
   "Seed ensemble" cell's note on `ARTIFACT_ROOT`.

In [ ]:
import subprocess
import zipfile
from pathlib import Path

from ffmodel.eval.run import discover_ensemble_roots

# Walk-forward bake-off: fail loudly here -- there's nothing meaningful to
# commit if this doesn't succeed. Auto-discovers any complete seed-ensemble
# roots (v1_s43, v1_s44, ...) next to the base run -- see the "Seed
# ensemble (optional)" cell above. If that cell was skipped, this just
# evaluates the single main v1 run, so the bake-off always scores exactly
# what could deploy.
roots = discover_ensemble_roots(Path("models/transformer/v1"))
print(f"bake-off roots: {[str(r) for r in roots]}")

root_args = []
for r in roots:
    root_args += ["--transformer-root", str(r)]

subprocess.run(
    ["python", "-m", "ffmodel.eval.run", *root_args,
     "--out", "models/backtests/bakeoff.json"],
    check=True,
)

subprocess.run(["git", "add", "models/transformer", "models/backtests", "configs"], check=False)
subprocess.run(
    ["git", "commit", "-m", "model: transformer v1 walk-forward artifacts + bake-off results"],
    check=False,
)  # non-zero (harmlessly) if there's nothing new to commit -- don't treat as fatal

pushed = subprocess.run(["git", "push"], check=False).returncode == 0

if not pushed:
    # Kaggle/Colab-proof: those platforms often can't push with your git
    # credentials. Zip up everything a human needs to commit locally
    # instead of losing the run.
    zip_path = Path("models_artifacts.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for src_dir in ("models/transformer", "models/backtests"):
            for f in Path(src_dir).rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to("."))
        for f in Path("models/sweeps").rglob("results.json"):
            zf.write(f, f.relative_to("."))
    print(
        f"git push failed -- wrote {zip_path.resolve()} instead.\n"
        "download this file and, in your local clone:\n"
        f"  unzip {zip_path.name}\n"
        "  git add models/transformer models/backtests models/sweeps/**/results.json\n"
        '  git commit -m "model: transformer v1 walk-forward artifacts + bake-off results"\n'
        "  git push"
    )
else:
    print("pushed successfully")